# Xarray-Spatial Preview: Memory-safe thumbnails of large rasters

When a raster is backed by dask (e.g., loaded lazily from Zarr or a stack of GeoTIFFs), calling `.compute()` to visualize it can blow up your memory. `xrspatial.preview()` downsamples the data to a target pixel size using block averaging, and the whole operation stays lazy until you ask for the result. Peak memory is bounded by the largest chunk plus the small output array.

### What you'll build

1. Create a large chunked dask raster with synthetic terrain
2. Generate a 500x500 thumbnail from a multi-gigabyte source
3. Compare preview sizes at different output resolutions
4. Use the `.xrs` accessor shorthand

![Preview preview](images/preview_preview.png)

**Jump to a section:**
[Large raster](#Large-dask-raster) | [Basic preview](#Basic-preview) | [Output resolution](#Output-resolution) | [Accessor syntax](#Accessor-syntax)

Standard imports plus `preview` from xrspatial.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import dask.array as da
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from xrspatial import preview

## Large dask raster

Create a chunked array of synthetic terrain so the notebook runs anywhere without external data files. The terrain uses layered sine waves to produce visible structure in the preview.

In [ ]:
# 20,000 x 20,000 chunked terrain (~1.6 GB float32)
size = 20000
chunk = 2000

# Build coordinate arrays as dask arrays, then compute terrain lazily
yy = da.from_delayed(
    __import__('dask').delayed(np.arange)(size, dtype=np.float32),
    shape=(size,), dtype=np.float32,
).rechunk(chunk)
xx = yy.copy()

# Outer product via broadcasting: terrain = f(y, x)
terrain = (
    200 * da.sin(xx[None, :] / 400) * da.cos(yy[:, None] / 500)
    + 100 * da.sin(yy[:, None] / 300)
    + 500
).astype(np.float32)

big = xr.DataArray(
    terrain.rechunk((chunk, chunk)),
    dims=['y', 'x'],
    coords={'y': np.arange(size, dtype=float), 'x': np.arange(size, dtype=float)},
)

print(f"Shape:      {big.shape[0]:,} x {big.shape[1]:,}")
print(f"Chunk size: {big.data.chunksize}")
print(f"Num chunks: {big.data.numblocks}")
print(f"Total size: {big.data.nbytes / 1e9:.2f} GB")
print(f"Dtype:      {big.dtype}")

## Basic preview

`preview()` builds a lazy coarsen-then-mean graph. Calling `.compute()` on the result materializes only the small output array -- the full raster is never held in memory at once.

In [ ]:
small = preview(big, width=500).compute()

print(f"Output shape: {small.shape}")
print(f"Output size:  {small.nbytes / 1e6:.1f} MB")
print(f"Reduction:    {big.data.nbytes / small.nbytes:.0f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
small.plot.imshow(ax=ax, cmap='terrain', add_colorbar=True,
                  cbar_kwargs={'label': 'Elevation'})
ax.set_title(f'500x500 preview of a {big.data.nbytes / 1e9:.1f} GB raster')
ax.set_axis_off()
plt.tight_layout()

## Output resolution

Different `width` values trade detail for speed. Smaller previews compute faster because fewer chunks need to be read and averaged.

In [ ]:
widths = [100, 250, 500]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, w in zip(axes, widths):
    thumb = preview(big, width=w).compute()
    thumb.plot.imshow(ax=ax, cmap='terrain', add_colorbar=False)
    ax.set_title(f'width={w}  ({thumb.shape[0]}x{thumb.shape[1]})')
    ax.set_axis_off()

plt.suptitle('Preview at different output resolutions', fontsize=14, y=1.02)
plt.tight_layout()

# Save preview image
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/preview_preview.png', bbox_inches='tight', dpi=120)

## Accessor syntax

You can also call `preview` directly on a DataArray via the `.xrs` accessor. The result is identical.

In [ ]:
import xrspatial  # registers .xrs accessor

accessor_result = big.xrs.preview(width=500).compute()
np.testing.assert_array_equal(accessor_result.values, small.values)
print(f"Accessor result shape: {accessor_result.shape}")
print("Accessor output matches function output exactly.")

<div class="alert alert-block alert-warning">
<b>Width vs. height.</b> The <code>width</code> parameter controls the output width in pixels. The height is computed automatically to preserve the aspect ratio of the input array. If the input is not evenly divisible by the coarsening factor, edge pixels are trimmed.
</div>

### References

- [Coarsening (xarray docs)](https://docs.xarray.dev/en/stable/user-guide/computation.html#coarsen)
- [xrspatial.preview API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.preview.html)